# Kentucky Road Closure Analysis

A companion Jupyter notebook for exploring the road-closure data. The hosted [website](https://chrislambert-ky.github.io/report-ky-roadclosures/) uses a single [Analysis Dashboard](../html/analysis_dashboard.html) with a Count/Duration selector. This notebook keeps its count and duration sections separate and adds county and point maps:

1. **Analysis by Count** - closures by year, month, district and county
2. **Analysis by Duration** - total closure hours by year, month, district and county
3. **County Map** - choropleth of county polygons, switchable by year
4. **Closure Map** - every individual closure on an OpenStreetMap basemap, with filters

**How to use it:** run the cells from top to bottom. The dropdowns and sliders only work while the notebook is running (they will not be interactive in a static preview such as GitHub). Click a dropdown to filter; the charts and maps redraw automatically. The website links county selection to its district and limits the county chart to that district; the notebook's district and county dropdowns remain independent.

**Data:** the report dataset created by `python python/etl_pipeline.py` (run from the repository root). See [data_dictionary.md](data_dictionary.md) for field definitions.

---

## 1. Setup and load data

In [ ]:
import html
from pathlib import Path
import json

import folium
import folium.plugins
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import to_hex

# Locate the data whether the notebook runs from the repo root or from python/
dataset_candidates = [
    Path("data-reportready/kytc-closures-report_dataset.csv"),
    Path("../data-reportready/kytc-closures-report_dataset.csv"),
]
dataset_path = next((p.resolve() for p in dataset_candidates if p.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError(
        "Run python python/etl_pipeline.py from the repository root to create "
        "data-reportready/kytc-closures-report_dataset.csv before running this notebook."
    )

geojson_candidates = [
    Path("python/reference/ky_counties.geojson"),
    Path("reference/ky_counties.geojson"),
]
geojson_path = next((p.resolve() for p in geojson_candidates if p.is_file()), None)
if geojson_path is None:
    raise FileNotFoundError("python/reference/ky_counties.geojson is missing.")

df = pd.read_csv(dataset_path, parse_dates=["Reported_On", "End_Date"])
df["Duration_Hours"] = pd.to_numeric(df["Duration_Hours"], errors="raise")
df["District"] = pd.to_numeric(df["District"], errors="coerce").astype("Int64")
df["Year"] = df["Reported_On"].dt.year
df["Month"] = df["Reported_On"].dt.month
df["Route_Prefix"] = df["Route"].astype(str).str.extract(r"^([A-Za-z]+)")[0].fillna("Unknown")

with open(geojson_path, encoding="utf-8") as f:
    ky_counties = json.load(f)

MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
YEARS = sorted(df["Year"].unique())
DISTRICTS = sorted(df["District"].dropna().unique())
COUNTIES = sorted(df["County"].dropna().unique())
ROUTE_PREFIXES = sorted(df["Route_Prefix"].unique())
KY_CENTER = [37.8, -85.7]

print(f"Loaded {len(df):,} closures from {dataset_path}")
print(f"{df['Reported_On'].min():%Y-%m-%d} to {df['Reported_On'].max():%Y-%m-%d}")
df.head()

## 2. Shared chart helpers

The website dashboard's four charts cross-filter each other: picking a year narrows the month, district and county charts, while the year chart keeps showing every year so you can still compare. Its district and county controls are linked: a district limits county results, and choosing a county selects its district. The notebook keeps these filters independent while applying them to its own charts.

In [ ]:
BAR_COLOR = "#003764"       # site's primary blue
HIGHLIGHT_COLOR = "#5EB3E4"  # site's selected-bar color
TOP_COUNTIES = 15


def apply_filters(frame, year=None, month=None, district=None, county=None, skip=None):
    """Filter closures; `skip` names a dimension to leave unfiltered."""
    if year is not None and skip != "year":
        frame = frame[frame["Year"] == year]
    if month is not None and skip != "month":
        frame = frame[frame["Month"] == month]
    if district is not None and skip != "district":
        frame = frame[frame["District"] == district]
    if county is not None and skip != "county":
        frame = frame[frame["County"] == county]
    return frame


def summarize(frame, key, metric):
    if metric == "count":
        return frame.groupby(key).size().astype(float)
    return frame.groupby(key)["Duration_Hours"].sum()


def filter_label(year, month, district, county):
    parts = []
    if year is not None:
        parts.append(f"Year {year}")
    if month is not None:
        parts.append(MONTH_NAMES[month - 1])
    if district is not None:
        parts.append(f"District {district}")
    if county is not None:
        parts.append(f"{county} County")
    return ", ".join(parts) if parts else "All closures"


def plot_dashboard(metric, year=None, month=None, district=None, county=None):
    """Draw the year / month / district / county charts for a metric ('count' or 'duration')."""
    axis_label = "Closures" if metric == "count" else "Total duration (hours)"
    fmt = "{:,.0f}"

    def colors_for(labels, selected):
        return [HIGHLIGHT_COLOR if label == selected else BAR_COLOR for label in labels]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    kwargs = dict(year=year, month=month, district=district, county=county)

    # Year (vertical bars)
    series = summarize(apply_filters(df, skip="year", **kwargs), "Year", metric).reindex(YEARS, fill_value=0)
    ax = axes[0, 0]
    bars = ax.bar([str(y) for y in series.index], series.values, color=colors_for(series.index, year))
    ax.bar_label(bars, labels=[fmt.format(v) for v in series.values], fontsize=8, padding=2)
    ax.set_title(f"{axis_label} by Year")
    ax.set_ylabel(axis_label)

    def hbar(ax, series, title, selected):
        bars = ax.barh([str(i) for i in series.index], series.values, color=colors_for(series.index, selected))
        ax.bar_label(bars, labels=[fmt.format(v) for v in series.values], fontsize=8, padding=2)
        ax.invert_yaxis()
        ax.set_title(title)
        ax.set_xlabel(axis_label)
        ax.margins(x=0.12)

    # Month
    series = summarize(apply_filters(df, skip="month", **kwargs), "Month", metric).reindex(range(1, 13), fill_value=0)
    series.index = MONTH_NAMES
    hbar(axes[0, 1], series, f"{axis_label} by Month", MONTH_NAMES[month - 1] if month else None)

    # District
    series = summarize(apply_filters(df, skip="district", **kwargs), "District", metric)
    series = series.reindex(DISTRICTS, fill_value=0)
    hbar(axes[1, 0], series, f"{axis_label} by District", district)

    # County (top N, always including the selected county)
    series = summarize(apply_filters(df, skip="county", **kwargs), "County", metric)
    series = series.sort_values(ascending=False)
    top = series.head(TOP_COUNTIES)
    if county is not None and county not in top.index:
        top = pd.concat([top, series.reindex([county], fill_value=0)])
    hbar(axes[1, 1], top, f"{axis_label} by County (top {TOP_COUNTIES})", county)

    filtered = apply_filters(df, **kwargs)
    total = len(filtered) if metric == "count" else filtered["Duration_Hours"].sum()
    unit = "closures" if metric == "count" else "hours"
    fig.suptitle(f"{filter_label(year, month, district, county)}  -  {total:,.0f} {unit}", fontsize=14, y=1.0)
    fig.tight_layout()
    plt.show()
    plt.close(fig)


def option_list(values, label=str):
    return [("All", None)] + [(label(v), v) for v in values]


def show_dashboard(metric):
    controls = {
        "year": widgets.Dropdown(options=option_list(YEARS), description="Year", layout=widgets.Layout(width="170px")),
        "month": widgets.Dropdown(
            options=[("All", None)] + [(name, i + 1) for i, name in enumerate(MONTH_NAMES)],
            description="Month", layout=widgets.Layout(width="170px")),
        "district": widgets.Dropdown(options=option_list(DISTRICTS), description="District", layout=widgets.Layout(width="190px")),
        "county": widgets.Dropdown(options=option_list(COUNTIES), description="County", layout=widgets.Layout(width="230px")),
    }
    reset = widgets.Button(description="Reset filters", button_style="")

    def on_reset(_):
        for control in controls.values():
            control.value = None

    reset.on_click(on_reset)
    output = widgets.interactive_output(lambda **kw: plot_dashboard(metric, **kw), controls)
    display(widgets.HBox(list(controls.values()) + [reset]), output)

## 3. Analysis by Count

Number of closures by year, month, district and county. Choose a value in any dropdown to filter the other charts (the selected bar turns light blue).

In [ ]:
show_dashboard("count")

## 4. Analysis by Duration

Total hours of closure by year, month, district and county. Same filters as above.

Durations are the time between the report and end timestamps, so a few very long closures can dominate a total. Compare with the count view above.

In [ ]:
show_dashboard("duration")

## 5. County map (choropleth by year)

Each Kentucky county is shaded by closure count or total closure hours. Pick a year to see how the pattern changes, or choose *All* for the full history. Hover over a county for its value.

In [ ]:
METRIC_OPTIONS = [("Closure count", "count"), ("Total closure hours", "duration")]


def build_county_map(metric="count", year=None):
    frame = df if year is None else df[df["Year"] == year]
    values = summarize(frame, "County", metric).reindex(COUNTIES, fill_value=0)
    legend = "Closures" if metric == "count" else "Total closure hours"
    title = f"{legend} by county - {'all years' if year is None else year}"

    # Copy the polygons and attach the values so the hover tooltip can show them
    geo = json.loads(json.dumps(ky_counties))
    for feature in geo["features"]:
        name = feature["properties"]["County"]
        feature["properties"]["Value"] = f"{values.get(name, 0):,.0f}"

    m = folium.Map(location=KY_CENTER, zoom_start=7, tiles="OpenStreetMap", control_scale=True)
    if values.max() > 0:
        positive = values[values > 0]
        bins = sorted({0.0, *np.quantile(positive, [0, 0.2, 0.4, 0.6, 0.8, 1.0])})
        table = pd.DataFrame({"County": values.index, "Value": values.to_numpy()})
        choropleth = folium.Choropleth(
            geo_data=geo,
            data=table,
            columns=["County", "Value"],
            key_on="feature.properties.County",
            fill_color="YlOrRd",
            fill_opacity=0.75,
            line_opacity=0.5,
            bins=bins,
            legend_name=title,
        ).add_to(m)
        choropleth.geojson.add_child(folium.GeoJsonTooltip(fields=["County", "Value"], aliases=["County", legend]))
    return m


def show_county_map():
    year = widgets.Dropdown(options=option_list(YEARS), description="Year", layout=widgets.Layout(width="170px"))
    metric = widgets.Dropdown(options=METRIC_OPTIONS, description="Metric", layout=widgets.Layout(width="260px"))
    output = widgets.interactive_output(lambda year, metric: display(build_county_map(metric, year)), {"year": year, "metric": metric})
    display(widgets.HBox([year, metric]), output)


show_county_map()

## 6. Closure map (individual points)

Every closure as a point on an OpenStreetMap basemap, colored by KYTC district. Use the controls to filter by year, month, district, county, route type (KY, US, I, ...) and minimum duration. Turn on **Cluster markers** to group nearby points when zoomed out. Click a point for details.

In [ ]:
_palette = plt.get_cmap("tab20")
DISTRICT_COLORS = {int(d): to_hex(_palette(i % 20)) for i, d in enumerate(DISTRICTS)}
DURATION_STEPS = [0, 1, 6, 12, 24, 72, 168, 720]


def popup_html(row):
    district = "n/a" if pd.isna(row.District) else int(row.District)
    return (
        f"<b>{html.escape(str(row.Road_Name))}</b> ({html.escape(str(row.Route))})<br>"
        f"{html.escape(str(row.County))} County, District {district}<br>"
        f"Reported: {row.Reported_On:%Y-%m-%d %H:%M}<br>"
        f"Duration: {row.Duration_Hours:,.1f} hours<br>"
        f"{html.escape(str(row.Comments))}"
    )


def build_points_map(year=None, month=None, district=None, county=None, route_prefix=None, min_hours=0, cluster=False):
    frame = apply_filters(df, year=year, month=month, district=district, county=county)
    if route_prefix is not None:
        frame = frame[frame["Route_Prefix"] == route_prefix]
    frame = frame[(frame["Duration_Hours"] >= min_hours) & frame["latitude"].notna() & frame["longitude"].notna()]

    m = folium.Map(location=KY_CENTER, zoom_start=7, tiles="OpenStreetMap", control_scale=True)
    if len(frame) and (county is not None or district is not None):
        m.fit_bounds([[frame["latitude"].min(), frame["longitude"].min()], [frame["latitude"].max(), frame["longitude"].max()]])

    target = folium.plugins.MarkerCluster().add_to(m) if cluster else m
    for row in frame.itertuples(index=False):
        color = DISTRICT_COLORS.get(int(row.District), "#666666") if pd.notna(row.District) else "#666666"
        folium.CircleMarker(
            location=[row.latitude, row.longitude],
            radius=5, color="#333333", weight=1, fill=True, fill_color=color, fill_opacity=0.85,
            popup=folium.Popup(popup_html(row), max_width=320),
        ).add_to(target)

    legend_rows = "".join(
        f'<div><span style="background:{c};display:inline-block;width:10px;height:10px;border:1px solid #333;"></span> District {d}</div>'
        for d, c in DISTRICT_COLORS.items()
    )
    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;bottom:25px;left:10px;z-index:9999;background:white;padding:6px 10px;'
        f'border:1px solid #999;border-radius:4px;font-size:11px;">{legend_rows}</div>'
    ))
    return m, len(frame)


def show_points_map():
    controls = {
        "year": widgets.Dropdown(options=option_list(YEARS), description="Year", layout=widgets.Layout(width="170px")),
        "month": widgets.Dropdown(options=[("All", None)] + [(n, i + 1) for i, n in enumerate(MONTH_NAMES)],
                                  description="Month", layout=widgets.Layout(width="170px")),
        "district": widgets.Dropdown(options=option_list(DISTRICTS), description="District", layout=widgets.Layout(width="190px")),
        "county": widgets.Dropdown(options=option_list(COUNTIES), description="County", layout=widgets.Layout(width="230px")),
        "route_prefix": widgets.Dropdown(options=option_list(ROUTE_PREFIXES), description="Route type", layout=widgets.Layout(width="200px")),
        "min_hours": widgets.SelectionSlider(options=DURATION_STEPS, value=0, description="Min hours",
                                             style={"description_width": "70px"}, layout=widgets.Layout(width="360px")),
        "cluster": widgets.Checkbox(value=False, description="Cluster markers", indent=False),
    }
    reset = widgets.Button(description="Reset filters")

    def on_reset(_):
        for name, control in controls.items():
            control.value = 0 if name == "min_hours" else False if name == "cluster" else None

    reset.on_click(on_reset)

    def draw(**kw):
        m, n = build_points_map(**kw)
        print(f"Showing {n:,} of {len(df):,} closures")
        display(m)

    output = widgets.interactive_output(draw, controls)
    rows = [
        widgets.HBox([controls["year"], controls["month"], controls["district"], controls["county"]]),
        widgets.HBox([controls["route_prefix"], controls["min_hours"], controls["cluster"], reset]),
    ]
    display(widgets.VBox(rows), output)


show_points_map()

## 7. Data quality check

The ETL keeps one record per location, comment, and report timestamp, preferring the record with the longest duration. This confirms the dataset has no duplicate events and shows how many records fall in each year.

In [ ]:
event_key = ["latitude", "longitude", "Comments", "Reported_On"]
print(f"Duplicate event keys: {df.duplicated(subset=event_key).sum():,}")
print("Records by report year:")
print(df.groupby("Year").size().to_string())